In [1]:
# ============================================================
# FASE 0 - BLOCCO 1: Setup ambiente (percorso corretto)
# ============================================================
import pandas as pd
import duckdb
from pathlib import Path

# --- Percorso base del progetto di gruppo ---
BASE_DIR = Path(r"C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism")

RAW_DIR = BASE_DIR / "data" / "raw"
STAGING_DIR = BASE_DIR / "data" / "staging"
DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"

for d in [RAW_DIR, STAGING_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Struttura cartelle creata dentro il progetto di gruppo:")
for d in [RAW_DIR, STAGING_DIR, DB_DIR]:
    print(f"  {d.resolve()}")

Struttura cartelle creata dentro il progetto di gruppo:
  C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw
  C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\staging
  C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\db


In [2]:
# ============================================================
# FASE 0 - BLOCCO 2: Connessione DB e schema
# ============================================================
con = duckdb.connect(str(DB_PATH))

con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE SCHEMA IF NOT EXISTS staging")
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")

# Verifica
schemi = con.execute("""
    SELECT schema_name 
    FROM information_schema.schemata 
    WHERE schema_name IN ('raw','staging','presentation')
""").df()
print(schemi)

    schema_name
0  presentation
1           raw
2       staging


In [3]:
# ============================================================
# FASE 0 - BLOCCO 3a: Download anagrafica comuni ISTAT (scoperta struttura)
# ============================================================
ISTAT_COMUNI_URL = "https://www.istat.it/storage/codici-unita-amministrative/Elenco-comuni-italiani.csv"

def scarica_anagrafica_comuni(url=ISTAT_COMUNI_URL):
    try:
        df = pd.read_csv(url, sep=';', encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(url, sep=';', encoding='cp1252')
    return df

df_comuni_italia = scarica_anagrafica_comuni()

print(f"Righe totali (comuni italiani): {len(df_comuni_italia)}")
print(f"\nColonne disponibili:")
for c in df_comuni_italia.columns:
    print(f"  - {c}")
print(f"\nPrime righe:")
df_comuni_italia.head(3)

Righe totali (comuni italiani): 7896

Colonne disponibili:
  - Codice Regione
  - Codice dell'Unità territoriale sovracomunale 
(valida a fini statistici)
  - Codice Provincia (Storico)(1)
  - Progressivo del Comune (2)
  - Codice Comune formato alfanumerico
  - Denominazione (Italiana e straniera)
  - Denominazione in italiano
  - Denominazione altra lingua
  - Codice Ripartizione Geografica
  - Ripartizione geografica
  - Denominazione Regione
  - Denominazione dell'Unità territoriale sovracomunale 
(valida a fini statistici)
  - Tipologia di Unità territoriale sovracomunale 
  - Flag Comune capoluogo di provincia/città metropolitana/libero consorzio
  - Sigla automobilistica
  - Codice Comune formato numerico
  - Codice Comune numerico con 110 province (dal 2010 al 2016)
  - Codice Comune numerico con 107 province (dal 2006 al 2009)
  - Codice Comune numerico con 103 province (dal 1995 al 2005)
  - Codice Catastale del comune
  - Codice NUTS1 2021
  - Codice NUTS2 2021 (3) 
  - Codi

,Codice Regione,Codice dell'Unità territoriale sovracomunale \n(valida a fini statistici),Codice Provincia (Storico)(1),Progressivo del Comune (2),Codice Comune formato alfanumerico,Denominazione (Italiana e straniera),Denominazione in italiano,Denominazione altra lingua,Codice Ripartizione Geografica,Ripartizione geografica,...,Codice Comune numerico con 110 province (dal 2010 al 2016),Codice Comune numerico con 107 province (dal 2006 al 2009),Codice Comune numerico con 103 province (dal 1995 al 2005),Codice Catastale del comune,Codice NUTS1 2021,Codice NUTS2 2021 (3),Codice NUTS3 2021,Codice NUTS1 2024,Codice NUTS2 2024 (3),Codice NUTS3 2024
0,1,201,1,1,1001,Agliè,Agliè,NaN,1,Nord-ovest,...,1001,1001,1001,A074,ITC,ITC1,ITC11,ITC,ITC1,ITC11
1,1,201,1,2,1002,Airasca,Airasca,NaN,1,Nord-ovest,...,1002,1002,1002,A109,ITC,ITC1,ITC11,ITC,ITC1,ITC11
2,1,201,1,3,1003,Ala di Stura,Ala di Stura,NaN,1,Nord-ovest,...,1003,1003,1003,A117,ITC,ITC1,ITC11,ITC,ITC1,ITC11


In [4]:
# ============================================================
# FASE 0 - BLOCCO 3b: Filtro Sardegna + tabella raw
# ============================================================

# Filtro sulla Denominazione Regione (più leggibile e robusto 
# rispetto a ricordare a memoria il codice numerico regione)
df_comuni_sardegna = df_comuni_italia[
    df_comuni_italia["Denominazione Regione"].str.strip() == "Sardegna"
].copy()

print(f"Comuni Sardegna trovati: {len(df_comuni_sardegna)}")

# Selezioniamo solo le colonne che ci servono davvero per il progetto,
# rinominandole in modo pulito per l'uso successivo (snake_case, italiano semplice)
colonne_utili = {
    "Codice Comune formato alfanumerico": "codice_istat_comune",
    "Codice Comune formato numerico": "codice_istat_numerico",
    "Denominazione in italiano": "comune",
    "Sigla automobilistica": "provincia_sigla",
    "Codice Provincia (Storico)(1)": "codice_provincia",
    "Ripartizione geografica": "ripartizione_geografica",
}

df_anagrafica = df_comuni_sardegna[list(colonne_utili.keys())].rename(columns=colonne_utili)

# Controllo qualità: nessun codice ISTAT duplicato o nullo (è la nostra chiave di join futura)
n_duplicati = df_anagrafica["codice_istat_comune"].duplicated().sum()
n_nulli = df_anagrafica["codice_istat_comune"].isna().sum()
print(f"Codici ISTAT duplicati: {n_duplicati}")
print(f"Codici ISTAT nulli: {n_nulli}")

df_anagrafica.head(10)


Comuni Sardegna trovati: 377
Codici ISTAT duplicati: 0
Codici ISTAT nulli: 0


,codice_istat_comune,codice_istat_numerico,comune,provincia_sigla,codice_provincia,ripartizione_geografica
7519,90001,90001,Aggius,SS,90,Isole
7520,90002,90002,Alà dei Sardi,SS,90,Isole
7521,90003,90003,Alghero,SS,90,Isole
7522,90004,90004,Anela,SS,90,Isole
7523,90005,90005,Ardara,SS,90,Isole
7524,90006,90006,Arzachena,SS,90,Isole
7525,90007,90007,Banari,SS,90,Isole
7526,90008,90008,Benetutti,SS,90,Isole
7527,90009,90009,Berchidda,SS,90,Isole
7528,90010,90010,Bessude,SS,90,Isole


In [5]:
# ============================================================
# FASE 0 - BLOCCO 3c: Salvataggio in DuckDB (schema raw)
# ============================================================
con.execute("DROP TABLE IF EXISTS raw.anagrafica_comuni")
con.execute("CREATE TABLE raw.anagrafica_comuni AS SELECT * FROM df_anagrafica")

verifica = con.execute("""
    SELECT COUNT(*) AS n_comuni, 
           COUNT(DISTINCT codice_istat_comune) AS n_codici_unici
    FROM raw.anagrafica_comuni
""").df()
print(verifica)

con.execute("SELECT * FROM raw.anagrafica_comuni LIMIT 5").df()

   n_comuni  n_codici_unici
0       377             377


,codice_istat_comune,codice_istat_numerico,comune,provincia_sigla,codice_provincia,ripartizione_geografica
0,90001,90001,Aggius,SS,90,Isole
1,90002,90002,Alà dei Sardi,SS,90,Isole
2,90003,90003,Alghero,SS,90,Isole
3,90004,90004,Anela,SS,90,Isole
4,90005,90005,Ardara,SS,90,Isole


In [6]:
# ============================================================
# RE-RUN FASE 0 - Blocco 1: Setup ambiente
# ============================================================
import pandas as pd
import duckdb
from pathlib import Path

BASE_DIR = Path(r"C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism")
RAW_DIR = BASE_DIR / "data" / "raw"
STAGING_DIR = BASE_DIR / "data" / "staging"
DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"

for d in [RAW_DIR, STAGING_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Cartelle pronte.")

Cartelle pronte.


In [7]:
# ============================================================
# RE-RUN FASE 0 - Blocco 2: Connessione DuckDB
# ============================================================
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE SCHEMA IF NOT EXISTS staging")
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")
print("Connessione DB attiva.")

Connessione DB attiva.


In [8]:
# ============================================================
# FASE 1 - BLOCCO B: Porti + Aeroporti (caricamento e pulizia)
# ============================================================
import pandas as pd

# Percorso del file così come l'hai salvato nel repo
PATH_RAW_PORTI_AEROPORTI = RAW_DIR / "bollettino_arrivi_partenze(1).csv"

df_porti_aeroporti = pd.read_csv(PATH_RAW_PORTI_AEROPORTI, sep=',', encoding='utf-8')

# Conversione data (formato dd/mm/yyyy)
df_porti_aeroporti["data"] = pd.to_datetime(df_porti_aeroporti["data"], format="%d/%m/%Y")

print(f"Righe totali nel file originale: {len(df_porti_aeroporti)}")
print(f"Periodo coperto: {df_porti_aeroporti['data'].min().date()} — {df_porti_aeroporti['data'].max().date()}")
print(f"Scali presenti: {sorted(df_porti_aeroporti['nome'].unique())}")

# Filtro sugli anni richiesti: 2022-2026
df_filtrato = df_porti_aeroporti[df_porti_aeroporti["data"].dt.year >= 2022].copy()

print(f"\nRighe dopo filtro 2022-2026: {len(df_filtrato)}")
print(f"Periodo effettivo dopo filtro: {df_filtrato['data'].min().date()} — {df_filtrato['data'].max().date()}")

df_filtrato.head(10)

Righe totali nel file originale: 19865
Periodo coperto: 2019-01-01 — 2026-07-11
Scali presenti: ['Alghero', 'Arbatax', 'Cagliari', 'Golfo Aranci', 'Olbia', 'Porto Torres', 'Porto Vesme']

Righe dopo filtro 2022-2026: 12132
Periodo effettivo dopo filtro: 2022-01-01 — 2026-07-11


,data,porto/aeroporto,nome,arrivi,partenze
0,2026-07-11,Aeroporto,Olbia,14279,12967
1,2026-07-11,Aeroporto,Cagliari,12825,11796
2,2026-07-11,Aeroporto,Alghero,4614,4004
3,2026-07-10,Aeroporto,Olbia,16723,12357
4,2026-07-10,Aeroporto,Cagliari,12375,10942
5,2026-07-10,Aeroporto,Alghero,5187,4519
6,2026-07-09,Porto,Porto Torres,1565,1200
7,2026-07-09,Porto,Olbia,8985,7117
8,2026-07-09,Porto,Golfo Aranci,1082,1042
9,2026-07-09,Porto,Cagliari,664,764


In [9]:
# ============================================================
# FASE 1 - BLOCCO C: Controllo qualità dati
# ============================================================

# Verifica valori nulli
print("Valori nulli per colonna:")
print(df_filtrato.isnull().sum())

# Verifica righe duplicate (stessa data+scalo due volte sarebbe un problema)
duplicati = df_filtrato.duplicated(subset=["data", "porto/aeroporto", "nome"]).sum()
print(f"\nRighe duplicate (data+scalo): {duplicati}")

# Controllo completezza: quanti giorni distinti per ciascuno scalo
completezza = df_filtrato.groupby(["porto/aeroporto", "nome"])["data"].agg(
    n_giorni="count",
    prima_data="min",
    ultima_data="max"
).reset_index()
print("\nCompletezza per scalo:")
completezza

Valori nulli per colonna:
data               0
porto/aeroporto    0
nome               0
arrivi             0
partenze           0
dtype: int64

Righe duplicate (data+scalo): 0

Completezza per scalo:


,porto/aeroporto,nome,n_giorni,prima_data,ultima_data
0,Aeroporto,Alghero,1644,2022-01-01,2026-07-11
1,Aeroporto,Cagliari,1644,2022-01-01,2026-07-11
2,Aeroporto,Olbia,1635,2022-01-01,2026-07-11
3,Porto,Arbatax,970,2022-01-02,2026-07-09
4,Porto,Cagliari,1607,2022-01-01,2026-07-09
5,Porto,Golfo Aranci,1249,2022-01-01,2026-07-09
6,Porto,Olbia,1595,2022-01-01,2026-07-09
7,Porto,Porto Torres,1566,2022-01-01,2026-07-09
8,Porto,Porto Vesme,222,2025-01-01,2026-07-06


In [10]:
# ============================================================
# FASE 1 - BLOCCO C-bis: Dettaglio giorni mancanti per scalo
# ============================================================
import pandas as pd

def giorni_mancanti_per_scalo(df, tipo, nome_scalo, data_inizio=None, data_fine=None):
    subset = df[(df["porto/aeroporto"] == tipo) & (df["nome"] == nome_scalo)]
    
    d_inizio = data_inizio or subset["data"].min()
    d_fine = data_fine or subset["data"].max()
    
    tutte_le_date = pd.date_range(start=d_inizio, end=d_fine, freq="D")
    date_presenti = set(subset["data"])
    date_mancanti = sorted(set(tutte_le_date) - date_presenti)
    
    print(f"\n--- {tipo} {nome_scalo} ---")
    print(f"Periodo atteso: {d_inizio.date()} — {d_fine.date()} ({len(tutte_le_date)} giorni)")
    print(f"Giorni mancanti: {len(date_mancanti)}")
    
    if date_mancanti:
        df_mancanti = pd.DataFrame({"data_mancante": date_mancanti})
        df_mancanti["anno"] = df_mancanti["data_mancante"].dt.year
        df_mancanti["mese"] = df_mancanti["data_mancante"].dt.month
        
        riepilogo = df_mancanti.groupby(["anno", "mese"]).size().reset_index(name="n_giorni_mancanti")
        print("Distribuzione mancanze per anno/mese:")
        print(riepilogo.to_string(index=False))
    
    return date_mancanti

# Golfo Aranci - il caso più sospetto
mancanti_golfo_aranci = giorni_mancanti_per_scalo(df_filtrato, "Porto", "Golfo Aranci")

# Aeroporto Olbia - scarto minore, ma controlliamo comunque
mancanti_olbia_aeroporto = giorni_mancanti_per_scalo(df_filtrato, "Aeroporto", "Olbia")


--- Porto Golfo Aranci ---
Periodo atteso: 2022-01-01 — 2026-07-09 (1651 giorni)
Giorni mancanti: 402
Distribuzione mancanze per anno/mese:
 anno  mese  n_giorni_mancanti
 2022     1                  3
 2022     2                 17
 2022     3                 20
 2022     4                  1
 2022     5                  2
 2022     9                  4
 2022    10                  1
 2022    11                 16
 2022    12                 12
 2023     1                 13
 2023     2                 16
 2023     3                 22
 2023     4                  2
 2023     8                  3
 2023     9                  2
 2023    11                 16
 2023    12                 16
 2024     1                 18
 2024     2                 19
 2024     3                 17
 2024     4                  8
 2024     5                  3
 2024     9                  1
 2024    10                 11
 2024    11                 17
 2024    12                 10
 2025     1           

In [11]:
# ============================================================
# FASE 1 - BLOCCO D: Salvataggio pulito e carico in DuckDB (raw)
# ============================================================

# Salvataggio versione filtrata/pulita in RAW_DIR (nome semplice, come da convenzione scelta)
PATH_OUTPUT = RAW_DIR / "porti_aeroporti.csv"
df_filtrato.to_csv(PATH_OUTPUT, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT}")

# Carico in DuckDB, schema raw
con.execute("DROP TABLE IF EXISTS raw.porti_aeroporti")
con.execute(f"""
    CREATE TABLE raw.porti_aeroporti AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           MIN(data) AS data_min,
           MAX(data) AS data_max,
           COUNT(DISTINCT nome) AS n_scali
    FROM raw.porti_aeroporti
""").df()
print(verifica)

Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\porti_aeroporti.csv


   n_righe   data_min   data_max  n_scali
0    12132 2022-01-01 2026-07-11        7


In [12]:
# ============================================================
# FASE 1 - BLOCCO B: Arrivi/Presenze SIRED - ispezione 2025
# ============================================================
import pandas as pd

PATH_SIRED_2025 = RAW_DIR / "csv_opendata_comuni_2025.csv"

df_sired_2025 = pd.read_csv(PATH_SIRED_2025, sep=',', encoding='utf-8')

print(f"Righe totali: {len(df_sired_2025)}")
print(f"Comuni distinti: {df_sired_2025['comune'].nunique()}")
print(f"Mesi presenti: {sorted(df_sired_2025['mese'].unique(), key=str)}")
print(f"\nMacro-tipologie:")
print(df_sired_2025['macro-tipologia'].value_counts())

print(f"\nRighe con 'non disponibile' in mese o macro-tipologia:")
mask_non_disp = (df_sired_2025['mese'] == 'non disponibile') | (df_sired_2025['macro-tipologia'] == 'non disponibile')
print(f"  Totale righe: {mask_non_disp.sum()} ({mask_non_disp.sum()/len(df_sired_2025)*100:.2f}%)")
print(f"  Comuni coinvolti: {df_sired_2025[mask_non_disp]['comune'].nunique()}")
print(f"  Esempi di comuni coinvolti: {df_sired_2025[mask_non_disp]['comune'].unique()[:15]}")

print(f"\nProvenienze distinte: {df_sired_2025['provenienza'].nunique()}")

Righe totali: 125422
Comuni distinti: 323
Mesi presenti: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9', 'non disponibile']

Macro-tipologie:
macro-tipologia
Esercizi Extra-Alberghieri:Alloggi privati in affitto    44883
Esercizi Alberghieri                                     40149
Esercizi Extra-Alberghieri:Esercizi Complementari        39050
non disponibile                                           1340
Name: count, dtype: int64

Righe con 'non disponibile' in mese o macro-tipologia:
  Totale righe: 1340 (1.07%)
  Comuni coinvolti: 256
  Esempi di comuni coinvolti: <ArrowStringArray>
[          'Anela',       'Bonnanaro',         'Bonorva',     'Bortigiadas',
     'Calangianus',       'Cheremule',    'Codrongianos',           'Luras',
 'non disponibile',          'Martis',         'Oschiri',            'Ossi',
          'Padria',        'Perfugas',         'Ploaghe']
Length: 15, dtype: str

Provenienze distinte: 80


In [13]:
# ============================================================
# FASE 1 - BLOCCO C: Verifica righe con comune non attribuibile
# ============================================================

righe_comune_non_disp = df_sired_2025[df_sired_2025['comune'] == 'non disponibile']
print(f"Righe con comune = 'non disponibile': {len(righe_comune_non_disp)}")
print(f"Arrivi totali coinvolti: {righe_comune_non_disp['arrivi'].sum()}")
print(f"Presenze totali coinvolte: {righe_comune_non_disp['presenze'].sum()}")
righe_comune_non_disp.head(10)

Righe con comune = 'non disponibile': 11
Arrivi totali coinvolti: 24
Presenze totali coinvolte: 280


,anno,provincia,comune,mese,macro-tipologia,provenienza,arrivi,presenze
379,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Marche,1,3
8456,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Stati Uniti d'America,3,3
29264,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Francia,3,3
63133,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,1,6
76566,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Repubblica Ceca,1,3
80384,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,0,10
92130,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Toscana,4,8
103908,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,2,2
107918,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Romania,8,240
115954,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,0,0


In [14]:
# ============================================================
# FASE 1 - BLOCCO C: Verifica righe con comune non attribuibile
# ============================================================

righe_comune_non_disp = df_sired_2025[df_sired_2025['comune'] == 'non disponibile']
print(f"Righe con comune = 'non disponibile': {len(righe_comune_non_disp)}")
print(f"Arrivi totali coinvolti: {righe_comune_non_disp['arrivi'].sum()}")
print(f"Presenze totali coinvolte: {righe_comune_non_disp['presenze'].sum()}")
righe_comune_non_disp.head(10)

Righe con comune = 'non disponibile': 11
Arrivi totali coinvolti: 24
Presenze totali coinvolte: 280


,anno,provincia,comune,mese,macro-tipologia,provenienza,arrivi,presenze
379,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Marche,1,3
8456,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Stati Uniti d'America,3,3
29264,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Francia,3,3
63133,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,1,6
76566,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Repubblica Ceca,1,3
80384,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,0,10
92130,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Toscana,4,8
103908,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,2,2
107918,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Romania,8,240
115954,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,0,0


In [15]:
# ============================================================
# FASE 1 - BLOCCO D: Caricamento raw.arrivi_presenze_sired
# ============================================================

con.execute("DROP TABLE IF EXISTS raw.arrivi_presenze_sired")
con.execute(f"""
    CREATE TABLE raw.arrivi_presenze_sired AS 
    SELECT * FROM read_csv_auto('{PATH_SIRED_2025.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(arrivi) AS arrivi_totali,
           SUM(presenze) AS presenze_totali
    FROM raw.arrivi_presenze_sired
""").df()
print(verifica)

   n_righe  n_comuni  n_anni  arrivi_totali  presenze_totali
0   125422       323       1      5170093.0       21922765.0


In [16]:
# ============================================================
# FASE 1 - BLOCCO E: Armonizzazione e concatenazione Arrivi/Presenze
# ============================================================
import pandas as pd

file_per_anno = {
    2022: RAW_DIR / "csv_opendata_comuni_2022.csv",
    2023: RAW_DIR / "csv_opendata_comuni_2023.csv",
    2024: RAW_DIR / "csv_opendata_comuni_2024.csv",
    2025: RAW_DIR / "csv_opendata_comuni_2025.csv",
}

def carica_e_armonizza(path, anno):
    df = pd.read_csv(path, sep=',', encoding='utf-8')
    
    # Armonizza nomi colonna (il 2022 usa macro_tipologia con underscore)
    df.columns = [c.strip().lower().replace('_', '-') if c.strip().lower() == 'macro_tipologia' else c.strip() for c in df.columns]
    df = df.rename(columns={"macro_tipologia": "macro-tipologia"})
    
    # Armonizza casing del nome comune (Title Case ovunque)
    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    
    # Armonizza casing della macro-tipologia "non disponibile"
    df["macro-tipologia"] = df["macro-tipologia"].astype(str).str.strip()
    df.loc[df["macro-tipologia"].str.lower() == "non disponibile", "macro-tipologia"] = "non disponibile"
    
    # Armonizza colonna mese (assicura sia stringa, per coerenza con 'non disponibile')
    df["mese"] = df["mese"].astype(str).str.strip()
    
    df["anno_verifica"] = anno  # controllo incrociato con la colonna anno originale
    return df

liste_df = []
for anno, path in file_per_anno.items():
    df_anno = carica_e_armonizza(path, anno)
    print(f"{anno}: {len(df_anno)} righe, colonne: {df_anno.columns.tolist()}")
    liste_df.append(df_anno)

df_sired_completo = pd.concat(liste_df, ignore_index=True)

# Controllo coerenza: la colonna 'anno' originale deve combaciare con l'anno del file
disallineati = df_sired_completo[df_sired_completo["anno"].astype(str) != df_sired_completo["anno_verifica"].astype(str)]
print(f"\nRighe con anno disallineato: {len(disallineati)}")

df_sired_completo = df_sired_completo.drop(columns=["anno_verifica"])
print(f"\nTotale righe concatenate: {len(df_sired_completo)}")
print(f"Comuni distinti: {df_sired_completo['comune'].nunique()}")
print(f"Anni coperti: {sorted(df_sired_completo['anno'].unique())}")
df_sired_completo.groupby("anno")[["arrivi", "presenze"]].sum()

2022: 75448 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']
2023: 98370 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']
2024: 110498 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']
2025: 125422 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']

Righe con anno disallineato: 0

Totale righe concatenate: 409738
Comuni distinti: 331
Anni coperti: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,arrivi,presenze
anno,,
2022,3720188.0,16387935
2023,3904389.0,16337893
2024,4442111.0,18907165
2025,5170093.0,21922765


In [17]:
# ============================================================
# FASE 1 - BLOCCO E-bis: Salvataggio Arrivi/Presenze in raw
# ============================================================

# Escludiamo le righe con comune = 'non disponibile' (decisione presa insieme prima)
df_sired_pulito = df_sired_completo[df_sired_completo["comune"] != "Non Disponibile"].copy()
print(f"Righe escluse (comune non disponibile): {len(df_sired_completo) - len(df_sired_pulito)}")

PATH_OUTPUT_SIRED = RAW_DIR / "arrivi_presenze_sired.csv"
df_sired_pulito.to_csv(PATH_OUTPUT_SIRED, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT_SIRED}")

con.execute("DROP TABLE IF EXISTS raw.arrivi_presenze_sired")
con.execute(f"""
    CREATE TABLE raw.arrivi_presenze_sired AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_SIRED.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(arrivi) AS arrivi_totali,
           SUM(presenze) AS presenze_totali
    FROM raw.arrivi_presenze_sired
""").df()
print(verifica)

Righe escluse (comune non disponibile): 277
Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\arrivi_presenze_sired.csv
   n_righe  n_comuni  n_anni  arrivi_totali  presenze_totali
0   409461       330       4     17195421.0       73469531.0


In [18]:
# ============================================================
# FASE 1 - BLOCCO F: Armonizzazione e concatenazione Capacità Ricettiva (corretto)
# ============================================================
import pandas as pd

file_capacita_per_anno = {
    2022: RAW_DIR / "capacita_strutture_ricettive_annuale_2022.csv",
    2023: RAW_DIR / "capacita_strutture_ricettive_annuale_2023.csv",
    2024: RAW_DIR / "capacita_strutture_ricettive_annuale_2024.csv",
    2025: RAW_DIR / "capacita_strutture_ricettive_annuale_2025.csv",
}

def leggi_csv_con_fallback(path):
    try:
        return pd.read_csv(path, sep=',', encoding='utf-8')
    except UnicodeDecodeError:
        print(f"  -> {path.name}: non UTF-8, uso encoding cp1252")
        return pd.read_csv(path, sep=',', encoding='cp1252')

def carica_e_armonizza_capacita(path, anno):
    df = leggi_csv_con_fallback(path)
    
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={
        "stelle": "categoria",
        "numero strutture": "numero_strutture",
    })
    
    colonne_attese = ["anno", "provincia", "comune", "tipologia", "categoria", "numero_strutture", "letti", "camere"]
    mancanti = [c for c in colonne_attese if c not in df.columns]
    if mancanti:
        print(f"  ATTENZIONE anno {anno}: colonne mancanti rispetto allo schema comune: {mancanti}")
    
    df = df[[c for c in colonne_attese if c in df.columns]]
    
    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    df["categoria"] = df["categoria"].replace("NULL", pd.NA)
    
    df["anno_verifica"] = anno
    return df

liste_df_capacita = []
for anno, path in file_capacita_per_anno.items():
    df_anno = carica_e_armonizza_capacita(path, anno)
    print(f"{anno}: {len(df_anno)} righe")
    liste_df_capacita.append(df_anno)

df_capacita_completo = pd.concat(liste_df_capacita, ignore_index=True)

disallineati = df_capacita_completo[df_capacita_completo["anno"].astype(str) != df_capacita_completo["anno_verifica"].astype(str)]
print(f"\nRighe con anno disallineato: {len(disallineati)}")
df_capacita_completo = df_capacita_completo.drop(columns=["anno_verifica"])

print(f"\nTotale righe concatenate: {len(df_capacita_completo)}")
print(f"Comuni distinti: {df_capacita_completo['comune'].nunique()}")
print(f"\nRiepilogo per anno:")
df_capacita_completo.groupby("anno")[["letti", "camere", "numero_strutture"]].sum()

2022: 1876 righe
  -> capacita_strutture_ricettive_annuale_2023.csv: non UTF-8, uso encoding cp1252
2023: 1914 righe
2024: 2328 righe
2025: 2471 righe

Righe con anno disallineato: 0

Totale righe concatenate: 8589
Comuni distinti: 354

Riepilogo per anno:


,letti,camere,numero_strutture
anno,,,
2022,302183,117261,21491
2023,324181,122209,27048
2024,388843,156536,39525
2025,459983,187897,52147


In [19]:
# ============================================================
# FASE 1 - BLOCCO G: Salvataggio Capacità Ricettiva in raw
# ============================================================

PATH_OUTPUT_CAPACITA = RAW_DIR / "capacita_ricettiva.csv"
df_capacita_completo.to_csv(PATH_OUTPUT_CAPACITA, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT_CAPACITA}")

con.execute("DROP TABLE IF EXISTS raw.capacita_ricettiva")
con.execute(f"""
    CREATE TABLE raw.capacita_ricettiva AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_CAPACITA.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(letti) AS letti_totali
    FROM raw.capacita_ricettiva
""").df()
print(verifica)

Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\capacita_ricettiva.csv
   n_righe  n_comuni  n_anni  letti_totali
0     8589       354       4     1475190.0


In [20]:
# ============================================================
# FASE 1 - BLOCCO J: Pulizia abitazioni occupate/non occupate (corretto)
# ============================================================
import pandas as pd

PATH_ABITAZIONI = RAW_DIR / "Abitazioni occupate e non occupate - comuni.csv"

df_abitazioni_raw = pd.read_csv(
    PATH_ABITAZIONI, 
    sep=',', 
    encoding='utf-8-sig',  # gestisce il BOM a inizio file
    quotechar="'"          # gestisce le note con virgole racchiuse in apici singoli
)

print(f"Righe totali (tutti i livelli territoriali d'Italia): {len(df_abitazioni_raw)}")
print(f"Valori distinti INDICATOR: {df_abitazioni_raw['INDICATOR'].unique()}")
print(f"Anni disponibili: {sorted(df_abitazioni_raw['TIME_PERIOD'].unique())}")

# Isoliamo solo le righe di livello comunale (REF_AREA puramente numerico)
df_abitazioni_raw["e_comune"] = df_abitazioni_raw["REF_AREA"].astype(str).str.isdigit()

df_solo_comuni = df_abitazioni_raw[df_abitazioni_raw["e_comune"]].copy()
print(f"\nRighe che sembrano essere di livello comunale: {len(df_solo_comuni)}")
print(f"Comuni distinti (tutta Italia): {df_solo_comuni['Territorio'].nunique()}")

Righe totali (tutti i livelli territoriali d'Italia): 71041
Valori distinti INDICATOR: <ArrowStringArray>
['NUM_OCC_DW_AV', 'NUM_UNOCC_DW_AV', 'NUM_DW_AV']
Length: 3, dtype: str
Anni disponibili: [np.int64(2019), np.int64(2021), np.int64(2023)]

Righe che sembrano essere di livello comunale: 69853
Comuni distinti (tutta Italia): 7774


In [21]:
# ============================================================
# FASE 1 - BLOCCO K: Filtro Sardegna + pivot indicatori
# ============================================================

# Uniforma il nome comune per il join (stesso trattamento delle altre fonti)
df_solo_comuni["comune_pulito"] = df_solo_comuni["Territorio"].astype(str).str.strip().str.title()

# Lista comuni sardi dalla nostra anagrafica di riferimento
comuni_sardegna = set(df_anagrafica["comune"].str.strip().str.title())

df_abitazioni_sardegna = df_solo_comuni[df_solo_comuni["comune_pulito"].isin(comuni_sardegna)].copy()

print(f"Righe Sardegna: {len(df_abitazioni_sardegna)}")
print(f"Comuni sardi trovati nel file: {df_abitazioni_sardegna['comune_pulito'].nunique()} (su 377 attesi)")

# Comuni sardi NON trovati (utile per capire se il join ha problemi di naming)
comuni_mancanti = comuni_sardegna - set(df_abitazioni_sardegna["comune_pulito"].unique())
print(f"\nComuni sardi non trovati nel file abitazioni: {len(comuni_mancanti)}")
if comuni_mancanti:
    print(sorted(comuni_mancanti))

# Trasformiamo da formato lungo a largo: una riga per comune-anno, colonne separate per indicatore
df_abitazioni_wide = df_abitazioni_sardegna.pivot_table(
    index=["comune_pulito", "TIME_PERIOD"],
    columns="INDICATOR",
    values="Osservazione",
    aggfunc="first"
).reset_index()

df_abitazioni_wide.columns.name = None
df_abitazioni_wide = df_abitazioni_wide.rename(columns={
    "comune_pulito": "comune",
    "TIME_PERIOD": "anno",
    "NUM_OCC_DW_AV": "abitazioni_occupate",
    "NUM_UNOCC_DW_AV": "abitazioni_non_occupate",
    "NUM_DW_AV": "abitazioni_totali"
})

# Controllo di coerenza: occupate + non occupate deve tornare vicino al totale
df_abitazioni_wide["somma_controllo"] = df_abitazioni_wide["abitazioni_occupate"] + df_abitazioni_wide["abitazioni_non_occupate"]
df_abitazioni_wide["differenza"] = df_abitazioni_wide["abitazioni_totali"] - df_abitazioni_wide["somma_controllo"]

print(f"\nComuni-anno nel formato finale: {len(df_abitazioni_wide)}")
print(f"Differenze totale vs occupate+non occupate (dovrebbero essere ~0):")
print(df_abitazioni_wide["differenza"].describe())

df_abitazioni_wide.head(10)

Righe Sardegna: 3330
Comuni sardi trovati nel file: 369 (su 377 attesi)

Comuni sardi non trovati nel file abitazioni: 8
["Quartu Sant'Elena", "San Nicolò D'Arcidano", "Sant'Andrea Frius", "Sant'Anna Arresi", "Sant'Antioco", "Sant'Antonio Di Gallura", "Trinità D'Agultu E Vignola", "Villa Sant'Antonio"]

Comuni-anno nel formato finale: 1107
Differenze totale vs occupate+non occupate (dovrebbero essere ~0):
count    1107.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: differenza, dtype: float64


,comune,anno,abitazioni_totali,abitazioni_occupate,abitazioni_non_occupate,somma_controllo,differenza
0,Abbasanta,2019,1609,1076,533,1609,0
1,Abbasanta,2021,1610,1128,482,1610,0
2,Abbasanta,2023,1635,1149,486,1635,0
3,Aggius,2019,1024,612,412,1024,0
4,Aggius,2021,1026,622,404,1026,0
5,Aggius,2023,1038,629,409,1038,0
6,Aglientu,2019,3255,655,2600,3255,0
7,Aglientu,2021,3264,674,2590,3264,0
8,Aglientu,2023,3293,701,2592,3293,0
9,Aidomaggiore,2019,354,192,162,354,0


In [22]:
# ============================================================
# FASE 1 - BLOCCO K-bis: Normalizzazione apostrofi e verifica
# ============================================================

def normalizza_apostrofi(testo):
    if pd.isna(testo):
        return testo
    return str(testo).replace("’", "'").replace("‘", "'")

# Applichiamo la normalizzazione a entrambe le fonti prima del confronto
df_solo_comuni["comune_pulito"] = df_solo_comuni["Territorio"].apply(normalizza_apostrofi).str.strip().str.title()
comuni_sardegna_norm = set(df_anagrafica["comune"].apply(normalizza_apostrofi).str.strip().str.title())

df_abitazioni_sardegna = df_solo_comuni[df_solo_comuni["comune_pulito"].isin(comuni_sardegna_norm)].copy()

print(f"Righe Sardegna (dopo normalizzazione apostrofi): {len(df_abitazioni_sardegna)}")
print(f"Comuni sardi trovati: {df_abitazioni_sardegna['comune_pulito'].nunique()} (su 377 attesi)")

comuni_mancanti_bis = comuni_sardegna_norm - set(df_abitazioni_sardegna["comune_pulito"].unique())
print(f"\nComuni sardi ancora non trovati: {len(comuni_mancanti_bis)}")
if comuni_mancanti_bis:
    print(sorted(comuni_mancanti_bis))

Righe Sardegna (dopo normalizzazione apostrofi): 3330
Comuni sardi trovati: 369 (su 377 attesi)

Comuni sardi ancora non trovati: 8
["Quartu Sant'Elena", "San Nicolò D'Arcidano", "Sant'Andrea Frius", "Sant'Anna Arresi", "Sant'Antioco", "Sant'Antonio Di Gallura", "Trinità D'Agultu E Vignola", "Villa Sant'Antonio"]


In [23]:
# ============================================================
# FASE 1 - BLOCCO K - ter
# ============================================================
import csv
import pandas as pd

# IMPORTANTE: usa il file CSV originale, quello scaricato la primissima volta
# da IstatData - NON il file .ods corretto a mano, NON un file già passato
# per pandas con quotechar="'"
PATH_ABITAZIONI = RAW_DIR / "Abitazioni occupate e non occupate - comuni.csv"

righe = []
with open(PATH_ABITAZIONI, encoding='utf-8-sig') as f:
    reader = csv.reader(f, delimiter=',', quoting=csv.QUOTE_NONE)
    header_completo = next(reader)
    colonne_necessarie = header_completo[:8]
    
    for riga in reader:
        righe.append(riga[:8])

df_abitazioni_raw = pd.DataFrame(righe, columns=colonne_necessarie)
df_abitazioni_raw["TIME_PERIOD"] = pd.to_numeric(df_abitazioni_raw["TIME_PERIOD"], errors="coerce")
df_abitazioni_raw["Osservazione"] = pd.to_numeric(df_abitazioni_raw["Osservazione"], errors="coerce")

# Solo comuni (REF_AREA numerico)
df_solo_comuni = df_abitazioni_raw[df_abitazioni_raw["REF_AREA"].astype(str).str.isdigit()].copy()

# Controllo apostrofi PRIMA di filtrare la Sardegna - deve essere pulito qui
controllo = df_solo_comuni[df_solo_comuni["Territorio"].str.contains("Sant'Antioco|Sant'Anna Arresi", case=False, na=False)]
print("Controllo apostrofi (deve essere pulito):")
for nome in controllo["Territorio"].unique():
    print(f"  {repr(nome)}")

Controllo apostrofi (deve essere pulito):


In [24]:
# ============================================================
# FASE 1 - BLOCCO K-quinto-bis: Diagnosi ampia su "Sant"
# ============================================================

righe_sant = df_solo_comuni[df_solo_comuni["Territorio"].str.contains("Sant", case=False, na=False)]
nomi_sant = sorted(righe_sant["Territorio"].unique())

print(f"Trovati {len(nomi_sant)} valori distinti con 'Sant':")
for nome in nomi_sant[:20]:
    print(f"  {repr(nome)}")

Trovati 206 valori distinti con 'Sant':
  '\'Aci Sant"\'Antonio\''
  '\'Albano Sant"\'Alessandro\''
  '\'Boschi Sant"\'Anna\''
  '\'Castel Sant"\'Angelo\''
  '\'Castel Sant"\'Elia\''
  '\'Castronuovo di Sant"\'Andrea\''
  '\'Cazzano Sant"\'Andrea\''
  '\'Città Sant"\'Angelo\''
  '\'Godega di Sant"\'Urbano\''
  '\'Isola Sant"\'Antonio\''
  '\'Mazzarrà Sant"\'Andrea\''
  '\'Monte Sant"\'Angelo\''
  '\'Mosciano Sant"\'Angelo\''
  '\'Motta Sant"\'Anastasia\''
  '\'Penna Sant"\'Andrea\''
  '\'Porto Sant"\'Elpidio\''
  '\'Quartu Sant"\'Elena\''
  '\'Rocchetta Sant"\'Antonio\''
  '\'Sant"\'Agapito\''
  '\'Sant"\'Agata Bolognese\''


In [25]:
# ============================================================
# FASE 1 - BLOCCO K-sesto: Correzione pattern apostrofo corrotto
# ============================================================
import re

def ripara_apostrofi(nome):
    if pd.isna(nome):
        return nome
    nome = str(nome)
    # Toglie un apostrofo "fantasma" a inizio e fine stringa, se presente
    if nome.startswith("'") and nome.endswith("'"):
        nome = nome[1:-1]
    # Sostituisce la sequenza corrotta "'  con un vero apostrofo
    nome = nome.replace('"\'', "'")
    return nome

df_solo_comuni["Territorio_riparato"] = df_solo_comuni["Territorio"].apply(ripara_apostrofi)

# Verifica sui casi noti
controllo = df_solo_comuni[df_solo_comuni["Territorio"].str.contains("Sant", case=False, na=False)]
print("Prima → Dopo la riparazione (primi 10 casi con 'Sant'):")
for _, row in controllo.head(10).iterrows():
    print(f"  {repr(row['Territorio'])}  →  {repr(row['Territorio_riparato'])}")

Prima → Dopo la riparazione (primi 10 casi con 'Sant'):
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Antonino di Susa\''  →  "Sant'Antonino di Susa"


In [26]:
# ============================================================
# FASE 1 - BLOCCO K-settimo: Filtro Sardegna + pivot (versione finale)
# ============================================================

# Usiamo la colonna riparata per il join
df_solo_comuni["comune_pulito"] = df_solo_comuni["Territorio_riparato"].astype(str).str.strip().str.title()

comuni_sardegna = set(df_anagrafica["comune"].str.strip().str.title())

df_abitazioni_sardegna = df_solo_comuni[df_solo_comuni["comune_pulito"].isin(comuni_sardegna)].copy()

print(f"Righe Sardegna: {len(df_abitazioni_sardegna)}")
print(f"Comuni sardi trovati: {df_abitazioni_sardegna['comune_pulito'].nunique()} (su 377 attesi)")

comuni_mancanti = comuni_sardegna - set(df_abitazioni_sardegna["comune_pulito"].unique())
print(f"\nComuni sardi ancora non trovati: {len(comuni_mancanti)}")
if comuni_mancanti:
    print(sorted(comuni_mancanti))

# Pivot in formato largo
df_abitazioni_wide = df_abitazioni_sardegna.pivot_table(
    index=["comune_pulito", "TIME_PERIOD"],
    columns="INDICATOR",
    values="Osservazione",
    aggfunc="first"
).reset_index()

df_abitazioni_wide.columns.name = None
df_abitazioni_wide = df_abitazioni_wide.rename(columns={
    "comune_pulito": "comune",
    "TIME_PERIOD": "anno",
    "NUM_OCC_DW_AV": "abitazioni_occupate",
    "NUM_UNOCC_DW_AV": "abitazioni_non_occupate",
    "NUM_DW_AV": "abitazioni_totali"
})

df_abitazioni_wide["somma_controllo"] = df_abitazioni_wide["abitazioni_occupate"] + df_abitazioni_wide["abitazioni_non_occupate"]
df_abitazioni_wide["differenza"] = df_abitazioni_wide["abitazioni_totali"] - df_abitazioni_wide["somma_controllo"]

print(f"\nComuni-anno nel formato finale: {len(df_abitazioni_wide)}")
print(f"Controllo coerenza (dovrebbe essere tutto 0):")
print(df_abitazioni_wide["differenza"].describe())

Righe Sardegna: 3402
Comuni sardi trovati: 377 (su 377 attesi)

Comuni sardi ancora non trovati: 0

Comuni-anno nel formato finale: 1131
Controllo coerenza (dovrebbe essere tutto 0):
count    1131.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: differenza, dtype: float64


In [27]:
# ============================================================
# FASE 1 - BLOCCO K-ottavo: Salvataggio Abitazioni in raw
# ============================================================

PATH_OUTPUT_ABITAZIONI = RAW_DIR / "abitazioni_non_occupate.csv"
df_abitazioni_wide.to_csv(PATH_OUTPUT_ABITAZIONI, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT_ABITAZIONI}")

con.execute("DROP TABLE IF EXISTS raw.abitazioni_non_occupate")
con.execute(f"""
    CREATE TABLE raw.abitazioni_non_occupate AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_ABITAZIONI.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(abitazioni_non_occupate) AS totale_non_occupate_sardegna,
           SUM(abitazioni_occupate) AS totale_occupate_sardegna
    FROM raw.abitazioni_non_occupate
""").df()
print(verifica)

Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\abitazioni_non_occupate.csv
   n_righe  n_comuni  n_anni  totale_non_occupate_sardegna  \
0     1131       377       3                      922452.0   

   totale_occupate_sardegna  
0                 2157984.0  


In [28]:
# ============================================================
# FASE 1 - BLOCCO L: Popolazione residente (2022-2025) e Superficie
# ============================================================
import pandas as pd

# --- Popolazione: 4 file, uno per anno ---
file_popolazione_per_anno = {
    2022: RAW_DIR / "Residenti 2022_Sardegna_totali_comune.csv",
    2023: RAW_DIR / "Residenti 2023_Sardegna_totali_comune.csv",
    2024: RAW_DIR / "Residenti 2024_Sardegna_totali_comune.csv",
    2025: RAW_DIR / "Residenti 2025_Sardegna_totali_comune.csv",
}

liste_popolazione = []
for anno, path in file_popolazione_per_anno.items():
    df = pd.read_csv(path, sep=',', encoding='utf-8-sig')
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"Comune": "comune", "Totale": "popolazione_residente"})
    df["comune"] = df["comune"].str.strip().str.title()
    df["anno"] = anno
    liste_popolazione.append(df)

df_popolazione = pd.concat(liste_popolazione, ignore_index=True)
print(f"Popolazione: {len(df_popolazione)} righe, {df_popolazione['comune'].nunique()} comuni, anni: {sorted(df_popolazione['anno'].unique())}")

# --- Superficie: 1 solo file (costante nel tempo) ---
PATH_SUPERFICIE = RAW_DIR / "Superficie_2025_Sardegna_totali_comune.csv"

df_superficie = pd.read_csv(PATH_SUPERFICIE, sep=',', encoding='utf-8-sig')
df_superficie.columns = [c.strip() for c in df_superficie.columns]
df_superficie = df_superficie.rename(columns={"Comune": "comune", "Superficie_kmq": "superficie_kmq"})
df_superficie["comune"] = df_superficie["comune"].str.strip().str.title()

# Conversione decimale italiano (virgola) -> numero
df_superficie["superficie_kmq"] = (
    df_superficie["superficie_kmq"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print(f"\nSuperficie: {len(df_superficie)} righe, {df_superficie['comune'].nunique()} comuni")
print(f"Superficie totale Sardegna: {df_superficie['superficie_kmq'].sum():.1f} kmq")

# Controllo comuni mancanti rispetto all'anagrafica (stesso controllo fatto altre volte)
comuni_sardegna = set(df_anagrafica["comune"].str.strip().str.title())
mancanti_pop = comuni_sardegna - set(df_popolazione["comune"].unique())
mancanti_sup = comuni_sardegna - set(df_superficie["comune"].unique())
print(f"\nComuni mancanti in popolazione: {len(mancanti_pop)} -> {sorted(mancanti_pop)}")
print(f"Comuni mancanti in superficie: {len(mancanti_sup)} -> {sorted(mancanti_sup)}")

Popolazione: 1508 righe, 377 comuni, anni: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Superficie: 377 righe, 377 comuni
Superficie totale Sardegna: 24109.9 kmq

Comuni mancanti in popolazione: 0 -> []
Comuni mancanti in superficie: 0 -> []


In [29]:
# ============================================================
# FASE 1 - BLOCCO L-bis: Salvataggio Popolazione e Superficie in raw
# ============================================================

# Popolazione
PATH_OUTPUT_POP = RAW_DIR / "popolazione_residente.csv"
df_popolazione.to_csv(PATH_OUTPUT_POP, index=False, encoding='utf-8')

con.execute("DROP TABLE IF EXISTS raw.popolazione_residente")
con.execute(f"""
    CREATE TABLE raw.popolazione_residente AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_POP.as_posix()}')
""")

# Superficie
PATH_OUTPUT_SUP = RAW_DIR / "superficie_comunale.csv"
df_superficie.to_csv(PATH_OUTPUT_SUP, index=False, encoding='utf-8')

con.execute("DROP TABLE IF EXISTS raw.superficie_comunale")
con.execute(f"""
    CREATE TABLE raw.superficie_comunale AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_SUP.as_posix()}')
""")

verifica = con.execute("""
    SELECT 
        (SELECT COUNT(*) FROM raw.popolazione_residente) AS righe_popolazione,
        (SELECT COUNT(DISTINCT comune) FROM raw.popolazione_residente) AS comuni_popolazione,
        (SELECT COUNT(*) FROM raw.superficie_comunale) AS righe_superficie,
        (SELECT SUM(superficie_kmq) FROM raw.superficie_comunale) AS superficie_totale
""").df()
print(verifica)

   righe_popolazione  comuni_popolazione  righe_superficie  superficie_totale
0               1508                 377               377          24109.945


In [30]:
# ============================================================
# FASE 2 - STAGING-A: Tabella di riferimento comuni
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.comuni_riferimento AS
    SELECT 
        codice_istat_comune,
        comune,
        provincia_sigla,
        ripartizione_geografica
    FROM raw.anagrafica_comuni
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_comuni, COUNT(DISTINCT comune) AS n_nomi_unici
    FROM staging.comuni_riferimento
""").df()
print(verifica)

con.execute("SELECT * FROM staging.comuni_riferimento LIMIT 5").df()

   n_comuni  n_nomi_unici
0       377           377


,codice_istat_comune,comune,provincia_sigla,ripartizione_geografica
0,90001,Aggius,SS,Isole
1,90002,Alà dei Sardi,SS,Isole
2,90003,Alghero,SS,Isole
3,90004,Anela,SS,Isole
4,90005,Ardara,SS,Isole


In [31]:
# ============================================================
# FASE 2 - STAGING-B: Verifica comuni con apostrofo
# ============================================================

comuni_sospetti = ["Villa Sant'Antonio", "Sant'Antonio Di Gallura", "San Nicolò D'Arcidano", "Trinità D'Agultu E Vignola", "Domus De Maria"]

for nome in comuni_sospetti:
    parola_chiave = nome.split()[-1]  # ultima parola, di solito la più distintiva
    check = con.execute(
        "SELECT DISTINCT comune FROM raw.arrivi_presenze_sired WHERE comune LIKE ?",
        [f"%{parola_chiave}%"]
    ).df()
    print(f"Cercando '{nome}' (parola chiave: '{parola_chiave}'):")
    print(check)
    print()
    

Cercando 'Villa Sant'Antonio' (parola chiave: 'Sant'Antonio'):
                    comune
0  Sant'Antonio Di Gallura

Cercando 'Sant'Antonio Di Gallura' (parola chiave: 'Gallura'):
                    comune
0     Santa Teresa Gallura
1  Sant'Antonio Di Gallura

Cercando 'San Nicolò D'Arcidano' (parola chiave: 'D'Arcidano'):
                  comune
0  San Nicolò D'Arcidano

Cercando 'Trinità D'Agultu E Vignola' (parola chiave: 'Vignola'):
                       comune
0  Trinità D'Agultu E Vignola

Cercando 'Domus De Maria' (parola chiave: 'Maria'):
                 comune
0        Domus De Maria
1  Santa Maria Coghinas



In [32]:
# ============================================================
# FASE 2 - STAGING-A-bis: Chiave di join normalizzata
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.comuni_riferimento AS
    SELECT 
        codice_istat_comune,
        comune,
        provincia_sigla,
        ripartizione_geografica,
        LOWER(TRIM(REPLACE(comune, '’', ''''))) AS chiave_comune
    FROM raw.anagrafica_comuni
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_comuni, COUNT(DISTINCT chiave_comune) AS n_chiavi_uniche
    FROM staging.comuni_riferimento
""").df()
print(verifica)

   n_comuni  n_chiavi_uniche
0       377              377


In [33]:
# ============================================================
# FASE 2 - STAGING-B-bis: Arrivi/Presenze (con chiave normalizzata)
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.arrivi_presenze_annuale AS
    SELECT 
        r.comune,  -- nome ufficiale dall'anagrafica, non quello grezzo di SIRED
        a.anno,
        SUM(a.arrivi) AS arrivi_totali,
        SUM(a.presenze) AS presenze_totali
    FROM raw.arrivi_presenze_sired a
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(a.comune, '’', ''''))) = r.chiave_comune
    GROUP BY r.comune, a.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni,
           SUM(arrivi_totali) AS arrivi_totali_sardegna, SUM(presenze_totali) AS presenze_totali_sardegna
    FROM staging.arrivi_presenze_annuale
""").df()
print(verifica)

mancanti = con.execute("""
    SELECT r.comune
    FROM staging.comuni_riferimento r
    LEFT JOIN staging.arrivi_presenze_annuale a ON r.comune = a.comune
    WHERE a.comune IS NULL
""").df()
print(f"\nComuni senza dati arrivi/presenze: {len(mancanti)}")
print(mancanti['comune'].tolist())

   n_righe  n_comuni  n_anni  arrivi_totali_sardegna  presenze_totali_sardegna
0     1106       330       4              17195421.0                73469531.0

Comuni senza dati arrivi/presenze: 47
['Asuni', 'San Basilio', 'Ittireddu', 'Semestene', 'Olzai', 'Zerfaliu', 'Cargeghe', 'Neoneli', 'Bulzi', 'Noragugume', 'Barrali', 'Giave', 'Soddì', 'Ortacesus', 'Setzu', 'Illorai', "Villa Sant'Antonio", 'Montresta', 'Borutta', 'Tadasuni', 'Siapiccia', 'Birori', 'Oniferi', 'Simala', 'Segariu', 'Siris', 'Gesico', 'Siligo', 'Gonnoscodina', 'Morgongiori', 'Senis', 'Tiana', 'Orotelli', 'San Nicolò Gerrei', 'Suelli', 'Nule', 'Lei', 'Bidonì', 'Genuri', 'Ussaramanna', 'Esporlatu', 'Romana', 'Onanì', 'Mogorella', 'Villanova Truschedu', 'Curcuris', 'Goni']


In [34]:
# ============================================================
# FASE 2 - STAGING-C: Capacità ricettiva aggregata per comune×anno
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.capacita_annuale AS
    SELECT 
        r.comune,
        c.anno,
        SUM(c.numero_strutture) AS numero_strutture_totali,
        SUM(c.letti) AS letti_totali,
        SUM(c.camere) AS camere_totali
    FROM raw.capacita_ricettiva c
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(c.comune, '’', ''''))) = r.chiave_comune
    GROUP BY r.comune, c.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni,
           SUM(letti_totali) AS letti_totali_sardegna
    FROM staging.capacita_annuale
""").df()
print(verifica)

mancanti_capacita = con.execute("""
    SELECT r.comune
    FROM staging.comuni_riferimento r
    LEFT JOIN staging.capacita_annuale c ON r.comune = c.comune
    WHERE c.comune IS NULL
""").df()
print(f"\nComuni senza dati capacità ricettiva: {len(mancanti_capacita)}")

   n_righe  n_comuni  n_anni  letti_totali_sardegna
0     1351       354       4              1475190.0

Comuni senza dati capacità ricettiva: 23


In [35]:
# ============================================================
# FASE 2 - STAGING-D: Popolazione, Superficie, Abitazioni
# ============================================================

# --- Popolazione ---
con.execute("""
    CREATE OR REPLACE TABLE staging.popolazione AS
    SELECT 
        r.comune,
        p.anno,
        p.popolazione_residente
    FROM raw.popolazione_residente p
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(p.comune, '’', ''''))) = r.chiave_comune
""")

# --- Superficie (nessun anno, costante) ---
con.execute("""
    CREATE OR REPLACE TABLE staging.superficie AS
    SELECT 
        r.comune,
        s.superficie_kmq
    FROM raw.superficie_comunale s
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(s.comune, '’', ''''))) = r.chiave_comune
""")

# --- Abitazioni non occupate ---
con.execute("""
    CREATE OR REPLACE TABLE staging.abitazioni AS
    SELECT 
        r.comune,
        CAST(a.anno AS INTEGER) AS anno,
        a.abitazioni_occupate,
        a.abitazioni_non_occupate,
        a.abitazioni_totali
    FROM raw.abitazioni_non_occupate a
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(a.comune, '’', ''''))) = r.chiave_comune
""")

# Verifica delle tre insieme
for tabella in ["popolazione", "superficie", "abitazioni"]:
    v = con.execute(f"SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni FROM staging.{tabella}").df()
    print(f"{tabella}: {v.iloc[0]['n_righe']} righe, {v.iloc[0]['n_comuni']} comuni")

popolazione: 1508 righe, 377 comuni
superficie: 377 righe, 377 comuni
abitazioni: 1131 righe, 377 comuni


In [36]:
# ============================================================
# FASE 2 - STAGING-F: Join finale - tabella presentation
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    WITH griglia AS (
        SELECT r.comune, anni.anno
        FROM staging.comuni_riferimento r
        CROSS JOIN (SELECT UNNEST([2022, 2023, 2024, 2025]) AS anno) anni
    ),
    abitazioni_ultimo_disponibile AS (
        -- Per ogni comune, usiamo il dato 2023 (l'ultimo disponibile) per gli anni 2024/2025
        SELECT comune, abitazioni_occupate, abitazioni_non_occupate, abitazioni_totali
        FROM staging.abitazioni
        WHERE anno = 2023
    )
    SELECT 
        g.comune,
        g.anno,
        p.popolazione_residente,
        s.superficie_kmq,
        ap.arrivi_totali,
        ap.presenze_totali,
        c.numero_strutture_totali,
        c.letti_totali,
        c.camere_totali,
        ab.abitazioni_occupate,
        ab.abitazioni_non_occupate,
        ab.abitazioni_totali
    FROM griglia g
    LEFT JOIN staging.popolazione p ON g.comune = p.comune AND g.anno = p.anno
    LEFT JOIN staging.superficie s ON g.comune = s.comune
    LEFT JOIN staging.arrivi_presenze_annuale ap ON g.comune = ap.comune AND g.anno = ap.anno
    LEFT JOIN staging.capacita_annuale c ON g.comune = c.comune AND g.anno = c.anno
    LEFT JOIN abitazioni_ultimo_disponibile ab ON g.comune = ab.comune
    ORDER BY g.comune, g.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni
    FROM presentation.indicatori_comune_anno
""").df()
print(verifica)

# Controllo completezza per colonna (quanti NULL per ciascuna variabile)
completezza = con.execute("""
    SELECT 
        SUM(CASE WHEN popolazione_residente IS NULL THEN 1 ELSE 0 END) AS null_popolazione,
        SUM(CASE WHEN superficie_kmq IS NULL THEN 1 ELSE 0 END) AS null_superficie,
        SUM(CASE WHEN arrivi_totali IS NULL THEN 1 ELSE 0 END) AS null_arrivi,
        SUM(CASE WHEN letti_totali IS NULL THEN 1 ELSE 0 END) AS null_letti,
        SUM(CASE WHEN abitazioni_non_occupate IS NULL THEN 1 ELSE 0 END) AS null_abitazioni
    FROM presentation.indicatori_comune_anno
""").df()
print(completezza)

con.execute("SELECT * FROM presentation.indicatori_comune_anno WHERE comune = 'Villasimius' ORDER BY anno").df()

   n_righe  n_comuni  n_anni
0     1508       377       4
   null_popolazione  null_superficie  null_arrivi  null_letti  null_abitazioni
0               0.0              0.0        402.0       157.0              0.0


,comune,anno,popolazione_residente,superficie_kmq,arrivi_totali,presenze_totali,numero_strutture_totali,letti_totali,camere_totali,abitazioni_occupate,abitazioni_non_occupate,abitazioni_totali
0,Villasimius,2022,3705,58.173,137933.0,753266.0,820.0,11887.0,4522.0,1941,4541,6482
1,Villasimius,2023,3689,58.173,133429.0,737623.0,827.0,11915.0,4503.0,1941,4541,6482
2,Villasimius,2024,3721,58.173,153737.0,824403.0,1290.0,14333.0,5530.0,1941,4541,6482
3,Villasimius,2025,3735,58.173,172783.0,916156.0,1793.0,17027.0,6765.0,1941,4541,6482


In [37]:
# ============================================================
# FASE 2 - STAGING-G: Indicatori derivati
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    SELECT 
        *,
        ROUND(presenze_totali / NULLIF(popolazione_residente, 0), 2) AS presenze_per_residente,
        ROUND(presenze_totali / NULLIF(superficie_kmq, 0), 1) AS presenze_per_kmq,
        ROUND(presenze_totali / NULLIF(letti_totali * 365.0, 0), 3) AS tasso_occupazione_media,
        ROUND(abitazioni_non_occupate * 100.0 / NULLIF(abitazioni_totali, 0), 1) AS quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno
""")

con.execute("""
    SELECT comune, anno, presenze_per_residente, presenze_per_kmq, 
           tasso_occupazione_media, quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Villasimius' 
    ORDER BY anno
""").df()

,comune,anno,presenze_per_residente,presenze_per_kmq,tasso_occupazione_media,quota_abitazioni_non_occupate_pct
0,Villasimius,2022,203.31,12948.7,0.174,70.1
1,Villasimius,2023,199.95,12679.8,0.170,70.1
2,Villasimius,2024,221.55,14171.6,0.158,70.1
3,Villasimius,2025,245.29,15748.8,0.147,70.1


In [38]:
# ============================================================
# FASE 2 - STAGING-F-bis: Abitazioni con anno più vicino per periodo
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    WITH griglia AS (
        SELECT r.comune, anni.anno
        FROM staging.comuni_riferimento r
        CROSS JOIN (SELECT UNNEST([2022, 2023, 2024, 2025]) AS anno) anni
    ),
    abitazioni_per_periodo AS (
        -- 2022 usa il dato censimento 2021 (il più vicino disponibile prima)
        -- 2023, 2024, 2025 usano il dato censimento 2023 (il più recente disponibile)
        SELECT 
            g.comune, 
            g.anno,
            ab.abitazioni_occupate,
            ab.abitazioni_non_occupate,
            ab.abitazioni_totali
        FROM griglia g
        LEFT JOIN staging.abitazioni ab 
            ON g.comune = ab.comune 
            AND ab.anno = CASE WHEN g.anno = 2022 THEN 2021 ELSE 2023 END
    )
    SELECT 
        g.comune,
        g.anno,
        p.popolazione_residente,
        s.superficie_kmq,
        ap.arrivi_totali,
        ap.presenze_totali,
        c.numero_strutture_totali,
        c.letti_totali,
        c.camere_totali,
        ab.abitazioni_occupate,
        ab.abitazioni_non_occupate,
        ab.abitazioni_totali,
        ROUND(ap.presenze_totali / NULLIF(p.popolazione_residente, 0), 2) AS presenze_per_residente,
        ROUND(ap.presenze_totali / NULLIF(s.superficie_kmq, 0), 1) AS presenze_per_kmq,
        ROUND(ap.presenze_totali / NULLIF(c.letti_totali * 365.0, 0), 3) AS tasso_occupazione_media,
        ROUND(ab.abitazioni_non_occupate * 100.0 / NULLIF(ab.abitazioni_totali, 0), 1) AS quota_abitazioni_non_occupate_pct
    FROM griglia g
    LEFT JOIN staging.popolazione p ON g.comune = p.comune AND g.anno = p.anno
    LEFT JOIN staging.superficie s ON g.comune = s.comune
    LEFT JOIN staging.arrivi_presenze_annuale ap ON g.comune = ap.comune AND g.anno = ap.anno
    LEFT JOIN staging.capacita_annuale c ON g.comune = c.comune AND g.anno = c.anno
    LEFT JOIN abitazioni_per_periodo ab ON g.comune = ab.comune AND g.anno = ab.anno
    ORDER BY g.comune, g.anno
""")

con.execute("""
    SELECT comune, anno, abitazioni_non_occupate, quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Villasimius' 
    ORDER BY anno
""").df()

,comune,anno,abitazioni_non_occupate,quota_abitazioni_non_occupate_pct
0,Villasimius,2022,4505,70.4
1,Villasimius,2023,4541,70.1
2,Villasimius,2024,4541,70.1
3,Villasimius,2025,4541,70.1


In [39]:
# ============================================================
# FASE 2 - STAGING-F-ter: Tabella completa con tutti gli indicatori
# ============================================================

# Coefficiente per stimare i posti letto "informali" dalle abitazioni non occupate
# Assunzione dichiarata: numero medio di persone per abitazione in Italia (dato ISTAT recente ~2.3)
COEFFICIENTE_PERSONE_PER_ABITAZIONE = 2.3

con.execute(f"""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    WITH griglia AS (
        SELECT r.comune, anni.anno
        FROM staging.comuni_riferimento r
        CROSS JOIN (SELECT UNNEST([2022, 2023, 2024, 2025]) AS anno) anni
    ),
    abitazioni_per_periodo AS (
        SELECT 
            g.comune, 
            g.anno,
            ab.abitazioni_occupate,
            ab.abitazioni_non_occupate,
            ab.abitazioni_totali
        FROM griglia g
        LEFT JOIN staging.abitazioni ab 
            ON g.comune = ab.comune 
            AND ab.anno = CASE WHEN g.anno = 2022 THEN 2021 ELSE 2023 END
    ),
    base AS (
        SELECT 
            g.comune,
            g.anno,
            p.popolazione_residente,
            s.superficie_kmq,
            ap.arrivi_totali,
            ap.presenze_totali,
            c.numero_strutture_totali,
            c.letti_totali,
            c.camere_totali,
            ab.abitazioni_occupate,
            ab.abitazioni_non_occupate,
            ab.abitazioni_totali
        FROM griglia g
        LEFT JOIN staging.popolazione p ON g.comune = p.comune AND g.anno = p.anno
        LEFT JOIN staging.superficie s ON g.comune = s.comune
        LEFT JOIN staging.arrivi_presenze_annuale ap ON g.comune = ap.comune AND g.anno = ap.anno
        LEFT JOIN staging.capacita_annuale c ON g.comune = c.comune AND g.anno = c.anno
        LEFT JOIN abitazioni_per_periodo ab ON g.comune = ab.comune AND g.anno = ab.anno
    ),
    base_con_lag AS (
        SELECT 
            *,
            LAG(presenze_totali) OVER (PARTITION BY comune ORDER BY anno) AS presenze_anno_precedente,
            LAG(letti_totali) OVER (PARTITION BY comune ORDER BY anno) AS letti_anno_precedente
        FROM base
    )
    SELECT 
        * EXCLUDE (presenze_anno_precedente, letti_anno_precedente),
        
        -- Pressione turistica di base
        ROUND(presenze_totali / NULLIF(popolazione_residente, 0), 2) AS presenze_per_residente,
        ROUND(presenze_totali / NULLIF(superficie_kmq, 0), 1) AS presenze_per_kmq,
        ROUND(presenze_totali / NULLIF(letti_totali * 365.0, 0), 3) AS tasso_occupazione_media,
        
        -- Comportamento turistico
        ROUND(presenze_totali / NULLIF(arrivi_totali, 0), 2) AS permanenza_media_giorni,
        
        -- Crescita anno su anno
        ROUND((presenze_totali - presenze_anno_precedente) * 100.0 / NULLIF(presenze_anno_precedente, 0), 1) AS crescita_presenze_yoy_pct,
        ROUND((letti_totali - letti_anno_precedente) * 100.0 / NULLIF(letti_anno_precedente, 0), 1) AS crescita_letti_yoy_pct,
        
        -- Struttura dell'offerta ricettiva
        ROUND(letti_totali / NULLIF(numero_strutture_totali, 0), 1) AS letti_per_struttura,
        ROUND(letti_totali / NULLIF(popolazione_residente, 0), 3) AS letti_per_residente,
        
        -- Seconde case / offerta informale
        ROUND(abitazioni_non_occupate * 100.0 / NULLIF(abitazioni_totali, 0), 1) AS quota_abitazioni_non_occupate_pct,
        ROUND(abitazioni_non_occupate * {COEFFICIENTE_PERSONE_PER_ABITAZIONE}, 0) AS posti_letto_informali_stimati,
        ROUND((letti_totali + abitazioni_non_occupate * {COEFFICIENTE_PERSONE_PER_ABITAZIONE}) / NULLIF(popolazione_residente, 0), 3) AS pressione_potenziale_totale,
        
        -- Contesto demografico
        ROUND(popolazione_residente / NULLIF(superficie_kmq, 0), 1) AS densita_abitanti_kmq
        
    FROM base_con_lag
    ORDER BY comune, anno
""")

verifica = con.execute("SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni FROM presentation.indicatori_comune_anno").df()
print(verifica)

con.execute("""
    SELECT * FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Villasimius' ORDER BY anno
""").df()

   n_righe  n_comuni
0     1508       377


,comune,anno,popolazione_residente,superficie_kmq,arrivi_totali,presenze_totali,numero_strutture_totali,letti_totali,camere_totali,abitazioni_occupate,...,tasso_occupazione_media,permanenza_media_giorni,crescita_presenze_yoy_pct,crescita_letti_yoy_pct,letti_per_struttura,letti_per_residente,quota_abitazioni_non_occupate_pct,posti_letto_informali_stimati,pressione_potenziale_totale,densita_abitanti_kmq
0,Villasimius,2022,3705,58.173,137933.0,753266.0,820.0,11887.0,4522.0,1893,...,0.174,5.46,NaN,NaN,14.5,3.208,70.4,10362.0,6.005,63.7
1,Villasimius,2023,3689,58.173,133429.0,737623.0,827.0,11915.0,4503.0,1941,...,0.170,5.53,-2.1,0.2,14.4,3.230,70.1,10444.0,6.061,63.4
2,Villasimius,2024,3721,58.173,153737.0,824403.0,1290.0,14333.0,5530.0,1941,...,0.158,5.36,11.8,20.3,11.1,3.852,70.1,10444.0,6.659,64.0
3,Villasimius,2025,3735,58.173,172783.0,916156.0,1793.0,17027.0,6765.0,1941,...,0.147,5.30,11.1,18.8,9.5,4.559,70.1,10444.0,7.355,64.2
